# Mini-LLaMA (V0.25)
Modern PyTorch implementation of a Large Language Model.

**Key Features:**
- **PyTorch** instead of TensorFlow/Keras
- **RoPE** (Rotary Position Embeddings)
- **RMSNorm** instead of LayerNorm
- **SwiGLU** activation in FFN
- **FlashAttention** via `F.scaled_dot_product_attention`
- **KV Cache** for O(N) inference
- **HuggingFace Datasets** (`roneneldan/TinyStories`) with sequence packing
- **Mixed Precision** (`bfloat16`) & proper AdamW weight decay

In [ ]:
!pip install -q torch transformers datasets tqdm

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Tuple
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader, IterableDataset

## 1. Architecture Components (LLaMA-style)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps)
        return self.weight * x

class SwiGLU(nn.Module):
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self.max_seq_len_cached = max_position_embeddings
        t = torch.arange(self.max_seq_len_cached, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, x, seq_len=None):
        if seq_len > self.max_seq_len_cached:
            self.max_seq_len_cached = seq_len
            t = torch.arange(self.max_seq_len_cached, device=x.device, dtype=self.inv_freq.dtype)
            freqs = torch.einsum("i,j->ij", t, self.inv_freq)
            emb = torch.cat((freqs, freqs), dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
            self.register_buffer("sin_cached", emb.sin()[None, None, :, :])
        return (
            self.cos_cached[:, :, :seq_len, ...].to(dtype=x.dtype),
            self.sin_cached[:, :, :seq_len, ...].to(dtype=x.dtype),
        )

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin, position_ids):
    cos = cos.squeeze(1).squeeze(0)  # [seq_len, dim]
    sin = sin.squeeze(1).squeeze(0)  # [seq_len, dim]
    cos = cos[position_ids].unsqueeze(1)  # [bs, 1, seq_len, dim]
    sin = sin[position_ids].unsqueeze(1)  # [bs, 1, seq_len, dim]
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

In [ ]:
class Attention(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.n_heads = args.n_heads
        self.n_kv_heads = args.n_kv_heads if args.n_kv_heads is not None else args.n_heads
        self.head_dim = args.dim // args.n_heads
        
        self.wq = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(args.dim, self.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(args.dim, self.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(args.n_heads * self.head_dim, args.dim, bias=False)
        
        self.rotary_emb = RotaryEmbedding(self.head_dim, max_position_embeddings=args.max_seq_len)

    def forward(self, x, position_ids, kv_cache=None):
        bsz, seqlen, _ = x.shape
        
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)
        
        xq = xq.view(bsz, seqlen, self.n_heads, self.head_dim).transpose(1, 2)
        xk = xk.view(bsz, seqlen, self.n_kv_heads, self.head_dim).transpose(1, 2)
        xv = xv.view(bsz, seqlen, self.n_kv_heads, self.head_dim).transpose(1, 2)
        
        cos, sin = self.rotary_emb(xv, seq_len=position_ids.max().item() + 1)
        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin, position_ids)
        
        if kv_cache is not None:
            k_cache, v_cache = kv_cache
            xk = torch.cat([k_cache, xk], dim=2)
            xv = torch.cat([v_cache, xv], dim=2)
        kv_cache = (xk, xv)
            
        # GQA / MQA repeat
        if self.n_kv_heads != self.n_heads:
            xk = torch.repeat_interleave(xk, self.n_heads // self.n_kv_heads, dim=1)
            xv = torch.repeat_interleave(xv, self.n_heads // self.n_kv_heads, dim=1)
            
        # FlashAttention
        output = F.scaled_dot_product_attention(xq, xk, xv, is_causal=True if seqlen > 1 else False)
        output = output.transpose(1, 2).contiguous().view(bsz, seqlen, -1)
        return self.wo(output), kv_cache

class TransformerBlock(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.attention = Attention(args)
        self.feed_forward = SwiGLU(args.dim, args.hidden_dim)
        self.attention_norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.ffn_norm = RMSNorm(args.dim, eps=args.norm_eps)

    def forward(self, x, position_ids, kv_cache=None):
        h, new_kv_cache = self.attention(self.attention_norm(x), position_ids, kv_cache)
        x = x + h
        x = x + self.feed_forward(self.ffn_norm(x))
        return x, new_kv_cache

In [ ]:
@dataclass
class ModelArgs:
    dim: int = 512
    n_layers: int = 8
    n_heads: int = 8
    n_kv_heads: int = None
    vocab_size: int = 50257 # GPT-2 vocab size
    hidden_dim: int = 1408 # multiple of 256
    max_seq_len: int = 1024
    norm_eps: float = 1e-5

class MiniLLaMA(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.vocab_size = args.vocab_size
        self.tok_embeddings = nn.Embedding(args.vocab_size, args.dim)
        self.layers = nn.ModuleList([TransformerBlock(args) for _ in range(args.n_layers)])
        self.norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.output = nn.Linear(args.dim, args.vocab_size, bias=False)
        
        # Weight tying
        self.tok_embeddings.weight = self.output.weight
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, tokens, position_ids=None, kv_caches=None):
        bsz, seqlen = tokens.shape
        if position_ids is None:
            position_ids = torch.arange(seqlen, dtype=torch.long, device=tokens.device).unsqueeze(0).expand(bsz, -1)
            
        h = self.tok_embeddings(tokens)
        
        new_kv_caches = []
        for i, layer in enumerate(self.layers):
            kv_cache = kv_caches[i] if kv_caches is not None else None
            h, new_kv_cache = layer(h, position_ids, kv_cache)
            new_kv_caches.append(new_kv_cache)
            
        h = self.norm(h)
        output = self.output(h)
        return output, new_kv_caches

    @torch.no_grad()
    def generate(self, prompt_tokens, max_gen_len, temperature=0.7, top_p=0.9):
        self.eval()
        device = next(self.parameters()).device
        tokens = torch.tensor([prompt_tokens], dtype=torch.long, device=device)
        
        kv_caches = None
        position_ids = torch.arange(tokens.shape[1], dtype=torch.long, device=device).unsqueeze(0)
        
        for _ in range(max_gen_len):
            logits, kv_caches = self(tokens if kv_caches is None else tokens[:, -1:], 
                                     position_ids if kv_caches is None else position_ids[:, -1:], 
                                     kv_caches)
            
            next_token_logits = logits[:, -1, :] / temperature
            
            # Top-p sampling
            sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
            next_token_logits[indices_to_remove] = -float('Inf')
            
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            tokens = torch.cat([tokens, next_token], dim=1)
            position_ids = torch.cat([position_ids, position_ids[:, -1:] + 1], dim=1)
            
        return tokens[0].tolist()

## 2. Data Pipeline (HuggingFace Datasets)

In [ ]:
# Using TinyStories for fast experimentation and coherent text generation
dataset = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M") # Fast BPE tokenizer
tokenizer.pad_token = tokenizer.eos_token

class PackedDataset(IterableDataset):
    def __init__(self, dataset, tokenizer, seq_len):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __iter__(self):
        buffer = []
        for sample in self.dataset:
            tokens = self.tokenizer(sample["text"], truncation=False, add_special_tokens=True)["input_ids"]
            buffer.extend(tokens)
            buffer.append(self.tokenizer.eos_token_id)
            
            while len(buffer) >= self.seq_len + 1:
                chunk = buffer[:self.seq_len + 1]
                buffer = buffer[self.seq_len:] # shift by seq_len
                yield torch.tensor(chunk, dtype=torch.long)

SEQ_LEN = 512
BATCH_SIZE = 8

train_dataset = PackedDataset(dataset, tokenizer, seq_len=SEQ_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)

## 3. Training Loop (Mixed Precision & AdamW)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

args = ModelArgs(vocab_size=tokenizer.vocab_size, max_seq_len=SEQ_LEN)
model = MiniLLaMA(args).to(device)

# Optimizer with weight decay fix (no decay on 1D tensors like Norm weights)
param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
optim_groups = [
    {'params': decay_params, 'weight_decay': 0.1},
    {'params': nodecay_params, 'weight_decay': 0.0}
]
optimizer = torch.optim.AdamW(optim_groups, lr=6e-4, betas=(0.9, 0.95))

scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')

# Training loop
model.train()
steps = 1000 # Adjust for longer training
pbar = tqdm(range(steps))
data_iter = iter(train_loader)

for step in pbar:
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        batch = next(data_iter)
        
    batch = batch.to(device)
    x = batch[:, :-1]
    y = batch[:, 1:]
    
    # Mixed Precision
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16):
        logits, _ = model(x)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        
    scaler.scale(loss).backward()
    
    # Gradient Clipping
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    
    if step % 10 == 0:
        pbar.set_description(f"Loss: {loss.item():.4f}")

## 4. Inference (O(N) with KV Cache)

In [ ]:
prompt = "Once upon a time, there was a little girl named Lily. She loved to"
prompt_tokens = tokenizer.encode(prompt)

print("Generating text...")
generated_tokens = model.generate(prompt_tokens, max_gen_len=50, temperature=0.8, top_p=0.9)
generated_text = tokenizer.decode(generated_tokens)

print("\n--- Generated Text ---")
print(generated_text)